# 1. Message Embedding

원본 Head Message/DR/Rule/Item 데이터를 통합하여 메시지 마스터 테이블을 만들고, BGE-M3 임베딩을 생성합니다.

## 입력 파일

- `data/raw/헤드메세지 MASTER 원본.xlsx`
- `data/raw/DR_Data.xlsx`
- `data/raw/Final_Rule_Deduction_Clean_add.xlsx`
- `data/raw/ITHM MASTER.xlsx`

## 출력 파일

- `data/processed/message_master_minimal.csv`
- `data/embeddings/message_embeddings_bge_m3_minimal.npy`
- `data/embeddings/embedding_meta_minimal.csv`


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import hashlib
from pathlib import Path
from datetime import datetime

from FlagEmbedding import BGEM3FlagModel

## 2. Path configuration

In [ ]:
# =========================
# Project path configuration
# =========================

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    "HEAD": RAW_DATA_DIR / "헤드메세지 MASTER 원본.xlsx",
    "DR": RAW_DATA_DIR / "DR_Data.xlsx",
    "RULE": RAW_DATA_DIR / "Final_Rule_Deduction_Clean_add.xlsx",
    "ITEM": RAW_DATA_DIR / "ITHM MASTER.xlsx",
}

MESSAGE_MASTER_PATH = PROCESSED_DATA_DIR / "message_master_minimal.csv"
MESSAGE_MASTER_PARQUET_PATH = PROCESSED_DATA_DIR / "message_master_minimal.parquet"
EMBEDDING_PATH = EMBEDDING_DIR / "message_embeddings_bge_m3_minimal.npy"
EMBEDDING_META_PATH = EMBEDDING_DIR / "embedding_meta_minimal.csv"

for name, path in RAW_FILES.items():
    if not path.exists():
        print(f"[WARN] {name} 원본 파일을 찾을 수 없습니다: {path}")

print("PROJECT_ROOT:", PROJECT_ROOT)

## 3. Raw data loading

In [ ]:
# =========================
# Load raw source files
# =========================

head_ms = pd.read_excel(RAW_FILES["HEAD"], engine="openpyxl")
DR = pd.read_excel(RAW_FILES["DR"], engine="openpyxl", header=1)
Rule = pd.read_excel(RAW_FILES["RULE"], engine="openpyxl", header=2)
item = pd.read_excel(RAW_FILES["ITEM"], engine="openpyxl")

In [ ]:
dfs = {
    "HEAD": head_ms,
    "DR": DR,
    "RULE": Rule,
    "ITEM": item
}

SOURCE_FILES = {name: path.name for name, path in RAW_FILES.items()}

def clean_column_name(col):
    col = str(col)
    col = col.replace("\n", " ")
    col = re.sub(r"\s+", " ", col)
    return col.strip()

for name, df in dfs.items():
    df.columns = [clean_column_name(c) for c in df.columns]

## 4. Column inspection and standardization

In [ ]:
for name, df in dfs.items():
    print("\n" + "=" * 100)
    print(f"[{name}] shape: {df.shape}")
    print("=" * 100)
    
    for i, col in enumerate(df.columns):
        print(f"{i:03d}: {col}")

In [ ]:
TEXT_KEYWORDS = [
    "message", "msg", "head", "trend", "comment", "issue", "reason",
    "action", "description", "remark", "text",
    "메시지", "메세지", "내용", "원인", "조치", "검토", "설명", "비고"
]

text_candidates = []

for source, df in dfs.items():
    for col in df.columns:
        col_lower = str(col).lower()
        
        if any(k.lower() in col_lower for k in TEXT_KEYWORDS):
            non_null = df[col].dropna().astype(str)
            
            text_candidates.append({
                "source_domain": source,
                "column_name": col,
                "non_null_count": len(non_null),
                "unique_count": non_null.nunique(),
                "sample_values": non_null.head(5).tolist()
            })

text_candidates_df = pd.DataFrame(text_candidates)
text_candidates_df

In [ ]:
DATE_KEYWORDS = [
    "date", "week", "기준", "날짜", "일자", "월", "주차"
]

date_candidates = []

for source, df in dfs.items():
    for col in df.columns:
        col_lower = str(col).lower()
        
        if any(k.lower() in col_lower for k in DATE_KEYWORDS):
            non_null = df[col].dropna()
            
            date_candidates.append({
                "source_domain": source,
                "column_name": col,
                "non_null_count": len(non_null),
                "sample_values": non_null.head(5).astype(str).tolist()
            })

date_candidates_df = pd.DataFrame(date_candidates)
date_candidates_df

In [ ]:
DATE_COLS = {
    "HEAD": "Date",   
    "DR": "Date",       
    "RULE": "Date",      
    "ITEM": "Date"       
}

MESSAGE_COLS = {
    "HEAD": [
        "Head Message",
        "Inventory Trend"
    ],
    "DR": [
        "Head Message"
    ],
    "RULE": [
        "HeadMessage"
    ],
    "ITEM": [
        "Head Message",
        "Inventory Trend"
    ]
}

## 5. Message master table construction

In [ ]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\n", " ")
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def is_valid_text(x):
    x = normalize_text(x)
    
    if x == "":
        return False
    
    if x.lower() in ["nan", "none", "null"]:
        return False
    
    if len(x) <= 1:
        return False
    
    # 숫자만 있는 값 제외
    if re.fullmatch(r"[-+]?\d+(\.\d+)?", x):
        return False
    
    return True

message_rows = []

for source, df in dfs.items():
    date_col = DATE_COLS.get(source)
    msg_cols = MESSAGE_COLS.get(source, [])
    
    print(f"\n[PROCESS] {source}")
    print(f"date_col: {date_col}")
    print(f"message_cols: {msg_cols}")
    
    for msg_col in msg_cols:
        if msg_col not in df.columns:
            print(f"[WARN] {source}에 컬럼 없음: {msg_col}")
            continue
        
        for idx, row in df.iterrows():
            message_text = normalize_text(row[msg_col])
            
            if not is_valid_text(message_text):
                continue
            
            if date_col is not None and date_col in df.columns:
                date_value = normalize_text(row[date_col])
            else:
                date_value = ""
            
            message_rows.append({
                "source_domain": source,
                "source_file": SOURCE_FILES[source],
                "row_idx_in_source": idx,
                "date": date_value,
                "message_type": msg_col,
                "message_text": message_text
            })

message_master = pd.DataFrame(message_rows)

message_master.insert(
    0,
    "message_id",
    [f"MSG_{i:06d}" for i in range(len(message_master))]
)

message_master["text_to_embed"] = (
    message_master["source_domain"].astype(str) + " " +
    message_master["message_type"].astype(str) + ": " +
    message_master["message_text"].astype(str)
)

message_master["text_to_embed"] = message_master["text_to_embed"].apply(normalize_text)

message_master["text_hash"] = message_master["text_to_embed"].apply(
    lambda x: hashlib.md5(x.encode("utf-8")).hexdigest()
)

message_master["embedding_row_idx"] = np.arange(len(message_master))

print(message_master.shape)
message_master.head()

In [ ]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

message_master.to_csv(
    MESSAGE_MASTER_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("saved:", MESSAGE_MASTER_PATH)

## 6. Save processed message table

In [ ]:
try:
    message_master.to_parquet(
        MESSAGE_MASTER_PARQUET_PATH,
        index=False
    )
    print("saved:", MESSAGE_MASTER_PARQUET_PATH)
except Exception as e:
    print("parquet 저장 실패:", e)

In [ ]:
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True) 

## 7. BGE-M3 embedding generation

In [ ]:
texts = message_master["text_to_embed"].tolist()

BATCH_SIZE = 8
MAX_LENGTH = 512

all_embeddings = []

for start in range(0, len(texts), BATCH_SIZE):
    end = start + BATCH_SIZE
    batch_texts = texts[start:end]
    
    output = model.encode(
        batch_texts,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH
    )
    
    batch_embeddings = output["dense_vecs"]
    all_embeddings.append(batch_embeddings)
    
    print(f"encoded {min(end, len(texts))} / {len(texts)}")

embeddings = np.vstack(all_embeddings).astype("float32")

print("embedding shape:", embeddings.shape)

In [ ]:
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.clip(norms, 1e-12, None)

embeddings_norm = embeddings / norms

print("first vector norm:", np.linalg.norm(embeddings_norm[0]))

In [ ]:
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

np.save(
    EMBEDDING_PATH,
    embeddings_norm
)

print("saved:", EMBEDDING_PATH)

## 8. Save and validate embeddings

In [ ]:
embedding_meta = message_master[[
    "message_id",
    "embedding_row_idx",
    "text_hash"
]].copy()

embedding_meta["embedding_model"] = "BAAI/bge-m3"
embedding_meta["embedding_dim"] = embeddings_norm.shape[1]
embedding_meta["max_length"] = MAX_LENGTH
embedding_meta["normalized"] = True
embedding_meta["created_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

embedding_meta.to_csv(
    EMBEDDING_META_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("saved:", EMBEDDING_META_PATH)

In [ ]:
message_master_loaded = pd.read_csv(MESSAGE_MASTER_PATH)
embeddings_loaded = np.load(EMBEDDING_PATH)

print(message_master_loaded.shape)
print(embeddings_loaded.shape)

assert len(message_master_loaded) == embeddings_loaded.shape[0]

In [ ]:
def embed_query(query_text):
    query_text = normalize_text(query_text)
    
    output = model.encode(
        [query_text],
        batch_size=1,
        max_length=512
    )
    
    query_emb = output["dense_vecs"].astype("float32")
    
    query_norm = np.linalg.norm(query_emb, axis=1, keepdims=True)
    query_norm = np.clip(query_norm, 1e-12, None)
    
    query_emb = query_emb / query_norm
    
    return query_emb[0]

def search_similar_messages(query_text, top_k=10):
    query_emb = embed_query(query_text)
    
    scores = embeddings_loaded @ query_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    result = message_master_loaded.iloc[top_indices].copy()
    result["similarity"] = scores[top_indices]
    
    return result[[
        "message_id",
        "source_domain",
        "source_file",
        "date",
        "message_type",
        "message_text",
        "similarity"
    ]]

## 9. Similarity search utility

In [ ]:
search_similar_messages("PO 수량 검토 필요", top_k=10)

In [ ]:
search_similar_messages("WIP 공정 과잉 자재 관리 필요", top_k=10)

In [ ]:
search_similar_messages("과잉 재고로 DIO 관리가 필요함", top_k=10)